# FinGenAI Module 05: Institutional-Grade Quantitative Risk Management Suite 📉🛡️

Welcome to **Module 5** of the **FinGenAI Masterclass**! In quantitative investing and algorithmic trading, profit generation is only half of the equation—**capital preservation and risk control** determine whether a quantitative strategy survives extreme market regimes.

This masterclass notebook introduces **5 world-class quantitative risk management frameworks**, fully implemented in pure Python within our localized `risk_management` package.

---

## 📐 Curriculum & Mathematical Index

| Risk Model | Mathematical Core | Key Quantitative Innovation |
| :--- | :--- | :--- |
| **1. Multi-Engine VaR & CVaR** | $\text{VaR}_\alpha = -\inf\{x \in \mathbb{R} : P(L > x) \le 1-\alpha\}$ | Fat-tailed Student-$t$, Filtered Historical Simulation (FHS-EWMA), Monte Carlo Cholesky, & EVT-GPD. |
| **2. Kelly Criterion & Position Sizing** | $f^* = \frac{p(b+1)-1}{b}$, $w^* = \mathbf{\Sigma}^{-1}\boldsymbol{\mu}$ | Merton Jump-Diffusion crash adjustments, Fractional Kelly, & Constrained Multi-Asset bounds. |
| **3. Volatility Target & Drawdown Control** | $\text{Leverage} = \min\left(\frac{\sigma_{target}}{\sigma_{realized}}, L_{max}\right)$ | Real-time Target Volatility scaling, Maximum Drawdown Circuit Breakers, & ATR Trailing Stops. |
| **4. Liquidity-Adjusted VaR & Stress Testing** | $\text{L-VaR} = \text{VaR} + \frac{1}{2} P_0 (\mu_s + z_\alpha \sigma_s)\sqrt{T_{liq}}$ | Exogenous bid-ask liquidation penalties, Regime Switching, & Historical Macro Crash Replay. |
| **5. Hierarchical Risk Parity (HRP)** | $d_{ij} = \sqrt{\frac{1}{2}(1 - \rho_{ij})}$ | Marcos López de Prado's HRP algorithm (Clustering, Quasi-Diagonalization, Recursive Bisection). |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import our institutional risk management package
from risk_management.var_cvar import VaRCVaREngine
from risk_management.kelly_sizing import KellySizingEngine
from risk_management.drawdown_vol_target import DrawdownVolTargetEngine, RiskState
from risk_management.liquidity_stress_regime import LiquidityStressRegimeEngine
from risk_management.hrp_risk_parity import HRPRiskParityEngine
from risk_management.engine import InstitutionalRiskManager

# Plotting style configuration
plt.style.use('seaborn-v0_8-darkgrid' if 'seaborn-v0_8-darkgrid' in plt.style.available else 'default')

--- 
## 1️⃣ Model 1: Value at Risk (VaR) & Expected Shortfall (CVaR)

Value at Risk (VaR) measures the maximum expected loss at a given confidence level over a specified time horizon. Expected Shortfall (CVaR) quantifies the expected loss *given that the loss exceeds the VaR threshold* (evaluating extreme tail risk).

### Mathematical Derivations:
1. **Parametric Gaussian VaR**:
$$\text{VaR}_{\alpha} = - (\mu \cdot h + z_{\alpha} \cdot \sigma \cdot \sqrt{h}) \cdot \text{Portfolio Value}$$

2. **Parametric Student-t Fat-Tailed VaR**:
$$\text{VaR}_{\alpha, \text{t}} = - (\mu \cdot h + t_{\alpha, \nu} \cdot s \cdot \sqrt{h}) \cdot \text{Portfolio Value}$$

3. **Extreme Value Theory (EVT-GPD)**:
$$\text{VaR}_{\alpha,\text{GPD}} = u + \frac{\beta}{\xi} \left[ \left( \frac{N}{N_u} (1 - \alpha) \right)^{-\xi} - 1 \right]$$

In [ ]:
# Generate synthetic multi-asset portfolio returns
np.random.seed(42)
n_days = 500
dates = pd.date_range("2024-01-01", periods=n_days, freq="B")

# Simulating stylized facts of asset returns (fat tails + clustering)
spy = np.random.standard_t(df=5, size=n_days) * 0.01 + 0.0004
qqq = 1.3 * spy + np.random.standard_t(df=4, size=n_days) * 0.008
tlt = -0.3 * spy + np.random.normal(0, 0.006, size=n_days)
gld = 0.1 * spy + np.random.normal(0, 0.007, size=n_days)

returns_df = pd.DataFrame({"SPY": spy, "QQQ": qqq, "TLT": tlt, "GLD": gld}, index=dates)
weights = np.array([0.4, 0.3, 0.2, 0.1])
port_returns = returns_df.values @ weights
portfolio_val = 1_000_000.0

# Run VaR/CVaR Engine
var_engine = VaRCVaREngine(confidence_level=0.95, time_horizon_days=1)
g_res = var_engine.parametric_gaussian(port_returns, portfolio_val)
t_res = var_engine.parametric_student_t(port_returns, portfolio_val)
h_res = var_engine.historical_simulation(port_returns, portfolio_val)
fhs_res = var_engine.filtered_historical_simulation(port_returns, portfolio_val)
mc_res = var_engine.monte_carlo_portfolio(returns_df, weights, portfolio_val)
evt_res = var_engine.extreme_value_theory_gpd(port_returns, portfolio_val)

print("=== 📊 95% 1-Day Value at Risk (VaR) & Expected Shortfall (CVaR) Comparison ===")
comparison_df = pd.DataFrame([
    {"Method": g_res["method"], "VaR ($)": g_res["dollar_var"], "CVaR ($)": g_res["dollar_cvar"]},
    {"Method": t_res["method"], "VaR ($)": t_res["dollar_var"], "CVaR ($)": t_res["dollar_cvar"]},
    {"Method": h_res["method"], "VaR ($)": h_res["dollar_var"], "CVaR ($)": h_res["dollar_cvar"]},
    {"Method": fhs_res["method"], "VaR ($)": fhs_res["dollar_var"], "CVaR ($)": fhs_res["dollar_cvar"]},
    {"Method": mc_res["method"], "VaR ($)": mc_res["dollar_var"], "CVaR ($)": mc_res["dollar_cvar"]},
    {"Method": evt_res["method"], "VaR ($)": evt_res["dollar_var"], "CVaR ($)": evt_res["dollar_cvar"]}
])
print(comparison_df.to_string(index=False))

--- 
## 2️⃣ Model 2: Kelly Criterion & Dynamic Position Sizing

The Kelly Criterion maximizes the expected logarithm of long-term wealth:
$$\max_f \mathbb{E}[\log(1 + f \cdot R)]$$

### Formulations:
1. **Continuous Log-Normal Kelly**:
$$f^* = \frac{\mu - r}{\sigma^2}$$

2. **Merton Jump-Diffusion Crash-Adjusted Kelly**:
$$\max_f \left\{ f (\mu - r - \lambda j) - \frac{1}{2} f^2 \sigma^2 + \lambda \log(1 + f j) \right\}$$

In [ ]:
# Execute Kelly Criterion calculations
b_kelly = KellySizingEngine.bernoulli_kelly(win_rate=0.56, win_loss_ratio=1.4, fractional_multiplier=0.5)
g_kelly = KellySizingEngine.continuous_gaussian_kelly(expected_return=0.15, volatility=0.20, fractional_multiplier=0.5)
j_kelly = KellySizingEngine.merton_jump_kelly(expected_return=0.15, diffusive_volatility=0.20, jump_intensity=0.5, mean_jump_size=-0.10)
m_kelly = KellySizingEngine.multi_asset_kelly(returns_df, fractional_multiplier=0.5)

print("=== 🎯 Kelly Position Sizing Outputs ===")
print(f"• Bernoulli Half-Kelly Fraction: {b_kelly['allocated_fraction']*100:.2f}%")
print(f"• Continuous Gaussian Half-Kelly Fraction: {g_kelly['allocated_fraction']*100:.2f}%")
print(f"• Merton Crash-Adjusted Jump Kelly Fraction: {j_kelly['allocated_fraction']*100:.2f}%")
print("• Multi-Asset Constrained Kelly Weights:", m_kelly['allocated_weights'])

--- 
## 3️⃣ Model 3: Volatility Targeting & Maximum Drawdown Circuit Breakers

Volatility targeting enforces constant risk profile regardless of market regimes. The Circuit Breaker state machine protects against catastrophic drawdowns.

```mermaid
graph LR
    NORMAL -->|DD >= 10%| WARNING
    WARNING -->|DD >= 15%| DELEVERAGING
    DELEVERAGING -->|DD >= 20%| HARD_HALT
    HARD_HALT -->|Peak Recovery| RECOVERY
    RECOVERY -->|Clean Benchmark| NORMAL
```

In [ ]:
# Run Volatility Targeting & Drawdown Engine
dd_engine = DrawdownVolTargetEngine(target_volatility=0.12, max_leverage=2.0)
vol_weight = dd_engine.calculate_volatility_target_weight(port_returns)

print("=== ⚡ Volatility Target Allocation ===")
print(f"• Target Volatility: {vol_weight['target_volatility']*100:.1f}%")
print(f"• Realized Volatility: {vol_weight['realized_volatility_annualized']*100:.1f}%")
print(f"• Dynamic Leverage Scalar: {vol_weight['leverage_scaling_factor']:.2f}x")

# Simulate Drawdown Circuit Breaker Transitions
print("\n=== 🚨 Circuit Breaker State Transitions ===")
equity_curve = [1_000_000, 1_150_000, 1_035_000, 977_500, 920_000, 910_000, 1_050_000]
for eq in equity_curve:
    status = dd_engine.update_circuit_breaker(eq)
    print(f"Portfolio Equity: ${eq:,.0f} | Drawdown: {status['current_drawdown_pct']} | State: {status['risk_state']} | Cap: {status['circuit_breaker_leverage_cap']}x")

--- 
## 4️⃣ Model 4: Liquidity-Adjusted VaR (L-VaR) & Macro Stress Testing

L-VaR incorporates bid-ask spread costs and exogenous liquidation time horizon $T_{liq}$:
$$\text{L-VaR} = \text{VaR}_{base} + \frac{1}{2} P_0 (\mu_{spread} + z_{\alpha} \sigma_{spread}) \sqrt{T_{liq}}$$

In [ ]:
# Run L-VaR & Macro Stress Testing Engine
spreads = np.random.uniform(0.0005, 0.003, size=len(port_returns))
lvar_res = LiquidityStressRegimeEngine.liquidity_adjusted_var(port_returns, spreads, portfolio_val, liquidation_horizon_days=5)
regime_res = LiquidityStressRegimeEngine.regime_aware_risk_adjustment(port_returns)
stress_df = LiquidityStressRegimeEngine.historical_macro_stress_test({"SPY": 0.4, "QQQ": 0.3, "TLT": 0.2, "GLD": 0.1}, portfolio_val)

print(f"=== 💧 L-VaR Audit ===")
print(f"• Base 5-Day VaR: ${lvar_res['base_dollar_var']:,.2f}")
print(f"• Liquidity Spread Cost: ${lvar_res['liquidity_spread_cost_dollar']:,.2f}")
print(f"• Total L-VaR: ${lvar_res['dollar_l_var']:,.2f}")
print(f"• Detected Market Volatility Regime: {regime_res['detected_regime']} (Scaling: {regime_res['regime_risk_scaling_multiplier']}x)")

print("\n=== 📉 Historical Macro Crash Stress Test Replay ===")
print(stress_df[['Scenario Name', 'Portfolio Return', 'Dollar Gain/Loss ($)', 'Post-Shock Portfolio Value ($)']].to_string(index=False))

--- 
## 5️⃣ Model 5: Hierarchical Risk Parity (HRP) & Covariance Shrinkage

Hierarchical Risk Parity (HRP) eliminates Markowitz covariance matrix inversion instability by combining graph-theoretic dendrogram clustering with recursive bisection allocation.

$$d_{ij} = \sqrt{\frac{1}{2} (1 - \rho_{ij})}$$

In [ ]:
# Execute HRP & Equal Risk Contribution (ERC)
hrp_engine = HRPRiskParityEngine()
hrp_res = hrp_engine.hierarchical_risk_parity(returns_df)
erc_res = hrp_engine.equal_risk_contribution(returns_df)

print("=== 🌳 Portfolio Weight Allocation Comparison ===")
alloc_df = pd.DataFrame({
    "Asset": list(returns_df.columns),
    "Hierarchical Risk Parity (HRP)": [f"{hrp_res['weights'][c]*100:.2f}%" for c in returns_df.columns],
    "Equal Risk Contribution (ERC)": [f"{erc_res['weights'][c]*100:.2f}%" for c in returns_df.columns]
})
print(alloc_df.to_string(index=False))

--- 
## 🏛️ Unified Master Institutional Risk Manager Audit

Finally, we run the master `InstitutionalRiskManager` gateway orchestrating all 5 modules in a single call.

In [ ]:
manager = InstitutionalRiskManager()
full_audit = manager.run_full_institutional_risk_audit(
    returns_df=returns_df,
    portfolio_weights=dict(zip(returns_df.columns, weights)),
    portfolio_value=portfolio_val,
    bid_ask_spreads=spreads
)

print("=== 🚀 MASTER INSTITUTIONAL RISK AUDIT COMPLETED ===")
print("• VaR/CVaR Summary:", full_audit['var_cvar_summary'])
print("• HRP Allocations:", full_audit['hrp_allocated_weights'])
print("• Circuit Breaker Status:", full_audit['volatility_target_status'])